In [1]:
# ==============================================================================
# STEP 0: AUTO-DOWNLOAD ALL NHANES FILES (run once, paste as first cell in Colab)
# ------------------------------------------------------------------------------
# Downloads Cycle J (2017-2018, PRIMARY — this is what your current model uses),
# plus Cycle I (2015-2016) and Cycle H (2013-2014) for later temporal validation.
# Files are saved into PROJECT_PATH so all your existing notebook cells work
# unchanged. Safe to re-run — skips files that already exist.
# ==============================================================================

import os
import requests

PROJECT_PATH = '/content/drive/MyDrive/MTLNN/'
os.makedirs(PROJECT_PATH, exist_ok=True)

# cycle_code -> (start_year_in_url, [component file stems])
CYCLES = {
    'J': {  # 2017-2018 — PRIMARY, matches your current paper/notebook
        'year': '2017',
        'files': ['DEMO_J', 'BMX_J', 'TRIGLY_J', 'HDL_J', 'BIOPRO_J',
                  'GLU_J', 'GHB_J', 'INS_J', 'BPX_J', 'LUX_J', 'DXX_J', 'ALQ_J']
    },
    'I': {  # 2015-2016 — for later temporal validation. NOTE: no LUX_I (liver
            # ultrasound component didn't exist yet — starts in Cycle J).
        'year': '2015',
        'files': ['DEMO_I', 'BMX_I', 'TRIGLY_I', 'HDL_I', 'BIOPRO_I',
                  'GLU_I', 'GHB_I', 'INS_I', 'BPX_I', 'DXX_I', 'ALQ_I']
    },
    'H': {  # 2013-2014 — for later temporal validation. Also no LUX_H.
        'year': '2013',
        'files': ['DEMO_H', 'BMX_H', 'TRIGLY_H', 'HDL_H', 'BIOPRO_H',
                  'GLU_H', 'GHB_H', 'INS_H', 'BPX_H', 'DXX_H', 'ALQ_H']
    },
}

BASE_URL = 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{year}/DataFiles/{stem}.xpt'

results = {'ok': [], 'skipped': [], 'failed': []}

for cycle_code, info in CYCLES.items():
    print(f"\n=== Cycle {cycle_code} ({info['year']}) ===")
    for stem in info['files']:
        out_path = os.path.join(PROJECT_PATH, f"{stem}.xpt")
        if os.path.exists(out_path):
            print(f"  ⏭️  {stem}.xpt already exists, skipping")
            results['skipped'].append(stem)
            continue

        url = BASE_URL.format(year=info['year'], stem=stem)
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            with open(out_path, 'wb') as f:
                f.write(r.content)
            size_kb = len(r.content) / 1024
            print(f"  ✅ {stem}.xpt downloaded ({size_kb:.0f} KB)")
            results['ok'].append(stem)
        except Exception as e:
            print(f"  ❌ {stem}.xpt FAILED — {e}")
            results['failed'].append(stem)

print("\n" + "=" * 60)
print(f"Done. {len(results['ok'])} downloaded, "
      f"{len(results['skipped'])} already present, "
      f"{len(results['failed'])} failed.")
if results['failed']:
    print(f"⚠️ Failed files (check these manually on the NHANES site): "
          f"{results['failed']}")
print("=" * 60)
print("\nAll files are now in:", PROJECT_PATH)
print("Cycle J files are what your current pipeline (Steps 2 onward) already uses.")
print("Cycle I and H files are staged for the later temporal-validation experiment —")
print("no need to touch them until we get to that step.")


=== Cycle J (2017) ===
  ✅ DEMO_J.xpt downloaded (3333 KB)
  ✅ BMX_J.xpt downloaded (1432 KB)
  ✅ TRIGLY_J.xpt downloaded (239 KB)
  ✅ HDL_J.xpt downloaded (175 KB)
  ✅ BIOPRO_J.xpt downloaded (2057 KB)
  ✅ GLU_J.xpt downloaded (96 KB)
  ✅ GHB_J.xpt downloaded (101 KB)
  ✅ INS_J.xpt downloaded (120 KB)
  ✅ BPX_J.xpt downloaded (1432 KB)
  ✅ LUX_J.xpt downloaded (678 KB)
  ✅ DXX_J.xpt downloaded (3729 KB)
  ✅ ALQ_J.xpt downloaded (434 KB)

=== Cycle I (2015) ===
  ✅ DEMO_I.xpt downloaded (3668 KB)
  ✅ BMX_I.xpt downloaded (1943 KB)
  ✅ TRIGLY_I.xpt downloaded (151 KB)
  ✅ HDL_I.xpt downloaded (189 KB)
  ✅ BIOPRO_I.xpt downloaded (2008 KB)
  ✅ GLU_I.xpt downloaded (101 KB)
  ✅ GHB_I.xpt downloaded (106 KB)
  ✅ INS_I.xpt downloaded (176 KB)
  ✅ BPX_I.xpt downloaded (1569 KB)
  ✅ DXX_I.xpt downloaded (4213 KB)
  ✅ ALQ_I.xpt downloaded (450 KB)

=== Cycle H (2013) ===
  ✅ DEMO_H.xpt downloaded (3743 KB)
  ✅ BMX_H.xpt downloaded (1998 KB)
  ✅ TRIGLY_H.xpt downloaded (158 KB)
  ✅ HDL_H.xpt d

# Multi-Target Metabolic Risk Prediction — V2 (Leakage-Controlled, Tiered)
Rebuilt pipeline. Fixes applied in this pass:
1. Real DXA-based obesity labels (age 18-59 sub-cohort) + honest BMI proxy for full cohort
2. Per-label leakage-free feature sets (not just Diabetes)
3. Split BEFORE imputation/Winsorization/scaling (no preprocessing leakage)
4. T0-T3 resource tiers (tape measure only -> full panel)
5. LR / RF / XGBoost baselines per tier

NN + missingness-degradation experiment: next pass, once these numbers look sane.

## Step 0: Mount Drive + Imports

In [2]:
from google.colab import drive
import os

drive.flush_and_unmount()
# Forcefully remove the mount point to ensure a clean state before remounting
!rm -rf /content/drive
drive.mount('/content/drive', force_remount=True)

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

PROJECT_PATH = '/content/drive/MyDrive/MTLNN/'
assert os.path.exists(PROJECT_PATH), "Folder not found — check PROJECT_PATH."
print("Connected:", PROJECT_PATH)
print("Files present:", sorted(os.listdir(PROJECT_PATH)))

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
Connected: /content/drive/MyDrive/MTLNN/
Files present: ['ALQ_H.xpt', 'ALQ_I.xpt', 'ALQ_J.xpt', 'BIOPRO_H.xpt', 'BIOPRO_I.xpt', 'BIOPRO_J.xpt', 'BMX_H.xpt', 'BMX_I.xpt', 'BMX_J.xpt', 'BPX_H.xpt', 'BPX_I.xpt', 'BPX_J.xpt', 'DEMO_H.xpt', 'DEMO_I.xpt', 'DEMO_J.xpt', 'DXX_H.xpt', 'DXX_I.xpt', 'DXX_J.xpt', 'GHB_H.xpt', 'GHB_I.xpt', 'GHB_J.xpt', 'GLU_H.xpt', 'GLU_I.xpt', 'GLU_J.xpt', 'HDL_H.xpt', 'HDL_I.xpt', 'HDL_J.xpt', 'INS_H.xpt', 'INS_I.xpt', 'INS_J.xpt', 'LUX_J.xpt', 'Multi_Target_Metabolic_Risk_Prediction_V2(F).ipynb', 'Multi_Target_Metabolic_Risk_Prediction_V3.ipynb', 'TRIGLY_H.xpt', 'TRIGLY_I.xpt', 'TRIGLY_J.xpt', 'dxa_subcohort_results.csv', 'missingness_degradation.csv', 'mtl_results.csv', 'mtl_vs_singletask.csv', 'tiered_results.csv']


## Step 1: Merge Cycle J files (core, specialty, DXA, alcohol)
Everything is LEFT-joined onto DEMO_J so we don't lose rows. DXA (`DXX_J`) will be sparse for ages 60+ by design — NHANES only scanned ages 8-59 — that's expected, not a bug.

In [3]:
core_files = {'demo': 'DEMO_J.xpt', 'bmx': 'BMX_J.xpt'}

specialty_files = {
    'trig':   'TRIGLY_J.xpt',
    'hdl':    'HDL_J.xpt',
    'biopro': 'BIOPRO_J.xpt',
    'glu':    'GLU_J.xpt',
    'ghb':    'GHB_J.xpt',
    'ins':    'INS_J.xpt',
    'bpx':    'BPX_J.xpt',
    'liver':  'LUX_J.xpt',
    'dxa':    'DXX_J.xpt',   # DXA body composition (ages 8-59 only)
    'alq':    'ALQ_J.xpt',   # alcohol questionnaire, for NAFLD refinement
}

df_raw = None
for key, filename in core_files.items():
    path = os.path.join(PROJECT_PATH, filename)
    temp = pd.read_sas(path)
    df_raw = temp if df_raw is None else pd.merge(df_raw, temp, on='SEQN', how='inner')

for key, filename in specialty_files.items():
    path = os.path.join(PROJECT_PATH, filename)
    if os.path.exists(path):
        temp = pd.read_sas(path)
        df_raw = pd.merge(df_raw, temp, on='SEQN', how='left')
    else:
        print(f"WARNING: {filename} not found, skipping ({key})")

print(f"Merged shape: {df_raw.shape}")


Merged shape: (8704, 259)


## Step 2: Adult filter + dedupe

In [4]:
df_raw = df_raw[df_raw['RIDAGEYR'] >= 18].reset_index(drop=True)
df_raw = df_raw.drop_duplicates(subset='SEQN').reset_index(drop=True)
print(f"Adults only: {df_raw.shape}")
print(f"Age range: {df_raw['RIDAGEYR'].min()}-{df_raw['RIDAGEYR'].max()}")
print(f"DXA (DXDTOPF) non-missing: {df_raw['DXDTOPF'].notna().sum()} "
      f"({df_raw['DXDTOPF'].notna().mean()*100:.1f}% of adults — expected to be low, DXA caps at age 59)")


Adults only: (5533, 259)
Age range: 18.0-80.0
DXA (DXDTOPF) non-missing: 2452 (44.3% of adults — expected to be low, DXA caps at age 59)


## Step 3: Column mapping

In [5]:
column_map = {
    'RIAGENDR':  'Gender',
    'RIDAGEYR':  'Age',
    'BMXBMI':    'BMI',
    'BMXWAIST':  'Waist',
    'BMXHT':     'Height',
    'LBXTR':     'Triglycerides',
    'LBDHDD':    'HDL',
    'LBXGLU':    'Glucose',
    'LBXGH':     'HbA1c',
    'LBXIN':     'Insulin',
    'BPXSY1':    'Systolic_BP',
    'BPXDI1':    'Diastolic_BP',
    'LBXSATSI':  'ALT',
    'LBXSASSI':  'AST',
    'LUXCAPM':   'Liver_Fat',
    'LBDLDL':    'LDL',
    'DXDTOPF':   'BodyFatPct_Total',   # DXA — ages 8-59 only
    'DXDTRPF':   'BodyFatPct_Trunk',   # DXA — ages 8-59 only
    'ALQ130':    'AlcoholDrinksPerDay',  # avg drinks/day past 12mo, for NAFLD refinement
}

available = {k: v for k, v in column_map.items() if k in df_raw.columns}
missing = set(column_map) - set(available)
if missing:
    print(f"NOTE: these source columns weren't found and are skipped: {missing}")

df_clean = df_raw.rename(columns=available)[list(available.values())].copy()
print(f"df_clean shape: {df_clean.shape}")
print(df_clean.isnull().sum())


df_clean shape: (5533, 19)
Gender                    0
Age                       0
BMI                      99
Waist                   349
Height                   89
Triglycerides          3040
HDL                     357
Glucose                2996
HbA1c                   272
Insulin                3048
Systolic_BP             634
Diastolic_BP            634
ALT                     380
AST                     396
Liver_Fat               414
LDL                    3065
BodyFatPct_Total       3081
BodyFatPct_Trunk       2919
AlcoholDrinksPerDay    2038
dtype: int64


## Step 4: Feature engineering (composite indices)
**No imputation, no Winsorization, no scaling here** — those all happen AFTER the train/test split in Step 6, fit on train only. This cell only computes deterministic formulas from raw values, which is safe to do before splitting.

In [6]:
df_clean['TyG'] = np.log((df_clean['Triglycerides'] * df_clean['Glucose']) / 2)

df_clean['WHtR'] = df_clean['Waist'] / df_clean['Height']

df_clean['HOMA_IR'] = (df_clean['Glucose'] / 18) * df_clean['Insulin'] / 22.5

vai_male = (df_clean['Waist'] / (39.68 + 1.88 * df_clean['BMI'])) * \
           (df_clean['Triglycerides'] / 1.03) * (1.31 / df_clean['HDL'])
vai_female = (df_clean['Waist'] / (36.58 + 1.89 * df_clean['BMI'])) * \
             (df_clean['Triglycerides'] / 0.81) * (1.52 / df_clean['HDL'])
df_clean['VAI'] = np.where(df_clean['Gender'] == 1, vai_male, vai_female)

lap_male = (df_clean['Waist'] - 65) * (df_clean['Triglycerides'] / 88.57)
lap_female = (df_clean['Waist'] - 58) * (df_clean['Triglycerides'] / 88.57)
df_clean['LAP'] = np.where(df_clean['Gender'] == 1, lap_male, lap_female)
df_clean['LAP'] = df_clean['LAP'].clip(lower=0)  # biologically can't be negative

print("Composite indices added:", ['TyG', 'WHtR', 'HOMA_IR', 'VAI', 'LAP'])
df_clean[['TyG', 'WHtR', 'HOMA_IR', 'VAI', 'LAP']].describe()


Composite indices added: ['TyG', 'WHtR', 'HOMA_IR', 'VAI', 'LAP']


,TyG,WHtR,HOMA_IR,VAI,LAP
count,2493.000000,5178.000000,2485.000000,2359.000000,2364.000000
mean,8.531945,0.602580,4.621094,4.118809,52.324516
std,0.701114,0.103013,9.341756,5.224417,59.415647
min,6.194405,0.366152,0.120963,0.229576,0.000000
25%,8.029107,0.530090,1.590469,1.712902,19.909394
50%,8.496174,0.595823,2.606247,2.912103,38.650220
75%,8.959376,0.667407,4.681481,4.999433,66.716721
max,12.308641,1.034913,178.810519,131.578196,1348.515299


## Step 5: Label definitions
**Two obesity/adiposity label sets, both honestly named:**
- `Label_HighBodyFat_BMI` / nothing fancier than a BMI cut — used for the PRIMARY, full 18-80 cohort. This is a proxy, not gold standard, and is documented as such.
- `Label_HighBodyFat_DXA` / `Label_AbdominalObesity_DXA` — real DXA thresholds, only valid for the ages-18-59 sub-cohort where `DXX_J` exists. Used for a separate validation analysis.

For NAFLD, we now exclude participants with significant alcohol intake where that data exists (ALQ130 >= 3 drinks/day is the conventional "significant alcohol use" cutoff for excluding alcoholic liver disease from a NAFLD proxy — participants above this are marked NaN, not silently kept, since we can't confidently proxy NAFLD for them).

In [7]:
df_clean['Label_HighBodyFat_BMI'] = (df_clean['BMI'] >= 30).astype(int)

# DXA-based labels — only defined where DXA data exists (ages 8-59)
male_hbf = (df_clean['Gender'] == 1) & (df_clean['BodyFatPct_Total'] >= 25)
fem_hbf  = (df_clean['Gender'] == 2) & (df_clean['BodyFatPct_Total'] >= 35)
df_clean['Label_HighBodyFat_DXA'] = np.where(
    df_clean['BodyFatPct_Total'].isna(), np.nan, (male_hbf | fem_hbf).astype(float))

male_ao = (df_clean['Gender'] == 1) & (df_clean['BodyFatPct_Trunk'] >= 34)
fem_ao  = (df_clean['Gender'] == 2) & (df_clean['BodyFatPct_Trunk'] >= 38)
df_clean['Label_AbdominalObesity_DXA'] = np.where(
    df_clean['BodyFatPct_Trunk'].isna(), np.nan, (male_ao | fem_ao).astype(float))

df_clean['Label_Low_HDL'] = (
    ((df_clean['Gender'] == 1) & (df_clean['HDL'] < 40)) |
    ((df_clean['Gender'] == 2) & (df_clean['HDL'] < 50))
).astype(int)

df_clean['Label_Diabetes'] = (
    (df_clean['HbA1c'] >= 6.5) | (df_clean['Glucose'] >= 126)
).astype(int)

# NAFLD proxy, with heavy-drinker exclusion where alcohol data is available
nafld_raw = ((df_clean['ALT'] > 40) | (df_clean['AST'] > 40)).astype(float)
heavy_drinker = df_clean['AlcoholDrinksPerDay'] >= 3
df_clean['Label_NAFLD'] = np.where(heavy_drinker.fillna(False), np.nan, nafld_raw)

df_clean['Label_Hypertension'] = (
    (df_clean['Systolic_BP'] >= 130) | (df_clean['Diastolic_BP'] >= 80)
).astype(int)

df_clean['Label_InsulinResistance'] = (df_clean['HOMA_IR'] >= 2.5).astype(int)

metsyn_abdominal = (
    ((df_clean['Gender'] == 1) & (df_clean['Waist'] > 102)) |
    ((df_clean['Gender'] == 2) & (df_clean['Waist'] > 88))
).astype(int)
metsyn_trig    = (df_clean['Triglycerides'] >= 150).astype(int)
metsyn_hdl     = df_clean['Label_Low_HDL']
metsyn_bp      = df_clean['Label_Hypertension']
metsyn_glucose = (df_clean['Glucose'] >= 100).astype(int)
df_clean['Label_MetSyn'] = (
    metsyn_abdominal + metsyn_trig + metsyn_hdl + metsyn_bp + metsyn_glucose >= 3
).astype(int)

PRIMARY_LABELS = [
    'Label_HighBodyFat_BMI', 'Label_Low_HDL', 'Label_Diabetes', 'Label_NAFLD',
    'Label_Hypertension', 'Label_InsulinResistance', 'Label_MetSyn',
]
DXA_LABELS = ['Label_HighBodyFat_DXA', 'Label_AbdominalObesity_DXA']

print("PRIMARY cohort label prevalence (full 18-80, n =", len(df_clean), "):")
for lab in PRIMARY_LABELS:
    valid = df_clean[lab].notna()
    n = df_clean.loc[valid, lab].sum()
    print(f"  {lab:<30} {int(n):>5} / {valid.sum():>5} ({100*n/valid.sum():.1f}%)")

print("\nDXA sub-cohort label prevalence (ages 18-59 only, valid DXA scan):")
for lab in DXA_LABELS:
    valid = df_clean[lab].notna()
    n = df_clean.loc[valid, lab].sum()
    print(f"  {lab:<30} {int(n):>5} / {valid.sum():>5} ({100*n/max(valid.sum(),1):.1f}%)")


PRIMARY cohort label prevalence (full 18-80, n = 5533 ):
  Label_HighBodyFat_BMI           2236 /  5533 (40.4%)
  Label_Low_HDL                   1531 /  5533 (27.7%)
  Label_Diabetes                   860 /  5533 (15.5%)
  Label_NAFLD                      324 /  4395 (7.4%)
  Label_Hypertension              2251 /  5533 (40.7%)
  Label_InsulinResistance         1301 /  5533 (23.5%)
  Label_MetSyn                    1190 /  5533 (21.5%)

DXA sub-cohort label prevalence (ages 18-59 only, valid DXA scan):
  Label_HighBodyFat_DXA           1713 /  2452 (69.9%)
  Label_AbdominalObesity_DXA       933 /  2614 (35.7%)


## Step 6: Per-label leakage-free feature exclusions
This is the core fix. Each label now excludes EVERY raw feature that is a direct component of its own definition — not just Diabetes. `ALL_FEATURES` is the full candidate pool; `LEAKAGE_EXCLUSIONS[label]` lists what must be dropped for that label specifically.

In [8]:
ALL_FEATURES = [
    'Gender', 'Age', 'BMI', 'Waist', 'Height',
    'Triglycerides', 'HDL', 'Glucose', 'HbA1c', 'Insulin',
    'Systolic_BP', 'Diastolic_BP', 'ALT', 'AST', 'Liver_Fat', 'LDL',
    'WHtR', 'VAI', 'LAP', 'HOMA_IR', 'TyG',
]

# Every feature that is a direct component of each label's own definition.
LEAKAGE_EXCLUSIONS = {
    'Label_HighBodyFat_BMI':      ['BMI'],
    'Label_HighBodyFat_DXA':      [],   # BodyFatPct_* aren't in ALL_FEATURES anyway
    'Label_AbdominalObesity_DXA': [],
    'Label_Low_HDL':              ['HDL'],
    'Label_Diabetes':             ['Glucose', 'HbA1c', 'TyG', 'HOMA_IR'],  # TyG/HOMA-IR both contain Glucose
    'Label_NAFLD':                ['ALT', 'AST', 'Liver_Fat'],
    'Label_Hypertension':         ['Systolic_BP', 'Diastolic_BP'],
    'Label_InsulinResistance':    ['HOMA_IR', 'Glucose', 'Insulin', 'TyG'],
    'Label_MetSyn':               ['Waist', 'Triglycerides', 'HDL', 'Systolic_BP',
                                    'Diastolic_BP', 'Glucose', 'WHtR', 'VAI', 'LAP',
                                    'TyG', 'HOMA_IR'],
}

def leakage_free_features(label, feature_pool=ALL_FEATURES):
    excl = set(LEAKAGE_EXCLUSIONS.get(label, []))
    return [f for f in feature_pool if f not in excl]

for lab, excl in LEAKAGE_EXCLUSIONS.items():
    kept = leakage_free_features(lab)
    print(f"{lab:<30} excludes {len(excl):>2} -> {len(kept)} features remain")


Label_HighBodyFat_BMI          excludes  1 -> 20 features remain
Label_HighBodyFat_DXA          excludes  0 -> 21 features remain
Label_AbdominalObesity_DXA     excludes  0 -> 21 features remain
Label_Low_HDL                  excludes  1 -> 20 features remain
Label_Diabetes                 excludes  4 -> 17 features remain
Label_NAFLD                    excludes  3 -> 18 features remain
Label_Hypertension             excludes  2 -> 19 features remain
Label_InsulinResistance        excludes  4 -> 17 features remain
Label_MetSyn                   excludes 11 -> 10 features remain


## Step 7: Resource tiers (T0-T3)
Each tier is a feature pool. The actual features used per (label, tier) combination is `leakage_free_features(label, tier_pool) ∩ tier_pool` — i.e., leakage exclusions still apply within a tier.

In [9]:
TIERS = {
    'T0_TapeMeasureOnly':   ['Gender', 'Age', 'BMI', 'Waist', 'Height', 'WHtR'],
    'T1_BasicLipids':       ['Gender', 'Age', 'BMI', 'Waist', 'Height', 'WHtR',
                              'Triglycerides', 'HDL', 'LDL', 'VAI', 'LAP'],
    'T2_ExtendedMetabolic': ['Gender', 'Age', 'BMI', 'Waist', 'Height', 'WHtR',
                              'Triglycerides', 'HDL', 'LDL', 'VAI', 'LAP',
                              'Glucose', 'HbA1c', 'Insulin', 'HOMA_IR', 'TyG'],
    # T3 = everything a standard blood panel + BP cuff + tape measure can produce.
    # Liver_Fat (LUXCAPM, transient elastography) is deliberately EXCLUDED here --
    # it requires a FibroScan device, which is not point-of-care and contradicts
    # the "no imaging required" framing of the paper. It gets its own tier below.
    'T3_ClinicalPanel':      [f for f in ALL_FEATURES if f != 'Liver_Fat'],
    # T4 = T3 + instrument-derived Liver_Fat. Kept separate so any AUC gain it buys
    # for NAFLD is reported honestly as "what you'd gain IF you also had ultrasound
    # access" rather than folded silently into the "full panel" number.
    'T4_EnhancedInstrument': ALL_FEATURES,
}

def tier_features_for_label(label, tier_name):
    pool = TIERS[tier_name]
    return leakage_free_features(label, pool)

# sanity check
for t in TIERS:
    print(t, '->', tier_features_for_label('Label_Diabetes', t))


T0_TapeMeasureOnly -> ['Gender', 'Age', 'BMI', 'Waist', 'Height', 'WHtR']
T1_BasicLipids -> ['Gender', 'Age', 'BMI', 'Waist', 'Height', 'WHtR', 'Triglycerides', 'HDL', 'LDL', 'VAI', 'LAP']
T2_ExtendedMetabolic -> ['Gender', 'Age', 'BMI', 'Waist', 'Height', 'WHtR', 'Triglycerides', 'HDL', 'LDL', 'VAI', 'LAP', 'Insulin']
T3_ClinicalPanel -> ['Gender', 'Age', 'BMI', 'Waist', 'Height', 'Triglycerides', 'HDL', 'Insulin', 'Systolic_BP', 'Diastolic_BP', 'ALT', 'AST', 'LDL', 'WHtR', 'VAI', 'LAP']
T4_EnhancedInstrument -> ['Gender', 'Age', 'BMI', 'Waist', 'Height', 'Triglycerides', 'HDL', 'Insulin', 'Systolic_BP', 'Diastolic_BP', 'ALT', 'AST', 'Liver_Fat', 'LDL', 'WHtR', 'VAI', 'LAP']


## Step 7b: Target-aware tier feature table
"T3" doesn't mean the same feature set for every label -- leakage exclusions remove different columns per label within the same tier pool. This table makes that explicit instead of leaving it implicit in `leakage_free_features()`, so nobody (including us, later) mistakes 'T3' for a single fixed feature list across all 9 labels.

In [10]:
tier_feature_matrix = []
for label in PRIMARY_LABELS + DXA_LABELS:
    row = {'Label': label.replace('Label_', '')}
    for tier_name in TIERS:
        feats = tier_features_for_label(label, tier_name)
        row[tier_name] = len(feats)
    tier_feature_matrix.append(row)

tier_feature_df = pd.DataFrame(tier_feature_matrix).set_index('Label')
print("=== Effective feature count per (label x tier), after leakage exclusions ===")
tier_feature_df


=== Effective feature count per (label x tier), after leakage exclusions ===


,T0_TapeMeasureOnly,T1_BasicLipids,T2_ExtendedMetabolic,T3_ClinicalPanel,T4_EnhancedInstrument
Label,,,,,
HighBodyFat_BMI,5,10,15,19,20
Low_HDL,6,10,15,19,20
Diabetes,6,11,12,16,17
NAFLD,6,11,16,18,18
Hypertension,6,11,16,18,19
InsulinResistance,6,11,12,16,17
MetSyn,4,5,7,9,10
HighBodyFat_DXA,6,11,16,20,21
AbdominalObesity_DXA,6,11,16,20,21


## Step 8: Train/test split (BEFORE any preprocessing)
Split happens on the raw, un-imputed, un-scaled data. Everything downstream fits only on `train_idx` rows and transforms both.

**Update:** now uses iterative multi-label stratification (`skmultilearn`) across all 7 primary labels instead of stratifying on `Label_Diabetes` alone, so rarer labels like NAFLD get proportional train/test representation too.

In [11]:
import subprocess, sys
try:
    from skmultilearn.model_selection import iterative_train_test_split
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-multilearn'], check=True)
    from skmultilearn.model_selection import iterative_train_test_split

all_labels = PRIMARY_LABELS + DXA_LABELS

# Multi-label-aware stratified split. Stratifying on Diabetes alone (the original
# approach) doesn't guarantee proportional representation of the other 6 primary
# labels -- NAFLD (7-9% prevalence) is the label most at risk of an unlucky split.
# Iterative stratification balances ALL primary labels simultaneously.
#
# NaN label values (NAFLD heavy-drinker exclusion) are filled with 0 for this
# stratification step ONLY -- it only decides which split a row lands in, it does
# NOT touch the actual label values used later for training/evaluation (those still
# come from df_clean/df_train_p/df_test_p with NaNs intact and properly excluded
# per-label in Step 10).
np.random.seed(42)
strat_label_cols = PRIMARY_LABELS  # DXA labels are age-gated (18-59 only) and handled
                                    # separately in Step 11, so they're excluded here
                                    # to avoid distorting the primary-cohort split.
Y_strat = df_clean[strat_label_cols].fillna(0).astype(int).values
X_idx = df_clean.index.values.reshape(-1, 1)

X_train_idx, _, X_test_idx, _ = iterative_train_test_split(X_idx, Y_strat, test_size=0.2)
train_idx = X_train_idx.flatten()
test_idx = X_test_idx.flatten()

df_train = df_clean.loc[train_idx].copy()
df_test  = df_clean.loc[test_idx].copy()
print(f"Train: {df_train.shape} | Test: {df_test.shape}")

print("\nPer-label prevalence check (train vs. test should now track closely):")
for lab in strat_label_cols:
    valid_tr = df_train[lab].notna()
    valid_te = df_test[lab].notna()
    tr = df_train.loc[valid_tr, lab].mean()
    te = df_test.loc[valid_te, lab].mean()
    print(f"  {lab:<30} train={tr:.3f}  test={te:.3f}  (diff={abs(tr-te):.3f})")


Train: (4426, 33) | Test: (1107, 33)

Per-label prevalence check (train vs. test should now track closely):
  Label_HighBodyFat_BMI          train=0.404  test=0.404  (diff=0.000)
  Label_Low_HDL                  train=0.277  test=0.276  (diff=0.000)
  Label_Diabetes                 train=0.155  test=0.155  (diff=0.000)
  Label_NAFLD                    train=0.074  test=0.074  (diff=0.000)
  Label_Hypertension             train=0.407  test=0.407  (diff=0.000)
  Label_InsulinResistance        train=0.235  test=0.235  (diff=0.000)
  Label_MetSyn                   train=0.213  test=0.222  (diff=0.009)


## Step 9: Preprocessing fit on TRAIN only (imputation + Winsorization)
Median and 1st/99th percentile values are computed from `df_train` only, then applied to both train and test. This is the fix for the preprocessing-leakage issue.

In [12]:
WINSORIZE_COLS = ['Triglycerides', 'Insulin', 'VAI', 'LAP', 'HOMA_IR', 'TyG']

def fit_preprocessing(df_train, cols_to_impute):
    medians = df_train[cols_to_impute].median(numeric_only=True)
    winsor_bounds = {}
    for col in WINSORIZE_COLS:
        if col in df_train.columns:
            winsor_bounds[col] = (df_train[col].quantile(0.01), df_train[col].quantile(0.99))
    return medians, winsor_bounds

def apply_preprocessing(df, medians, winsor_bounds):
    df = df.copy()
    for col in medians.index:
        if col in df.columns:
            df[col] = df[col].fillna(medians[col])
    for col, (lo, hi) in winsor_bounds.items():
        if col in df.columns:
            df[col] = df[col].clip(lo, hi)
    return df

impute_cols = ALL_FEATURES  # only impute the model-input features
medians, winsor_bounds = fit_preprocessing(df_train, impute_cols)

df_train_p = apply_preprocessing(df_train, medians, winsor_bounds)
df_test_p  = apply_preprocessing(df_test, medians, winsor_bounds)

print("Preprocessing fit on train, applied to both. Remaining NaNs in features (train):")
print(df_train_p[ALL_FEATURES].isnull().sum().sum())


Preprocessing fit on train, applied to both. Remaining NaNs in features (train):
0


## Step 10: Train LR / RF / XGBoost across all (label x tier) combinations
Reports F1 (weighted), ROC-AUC, and PR-AUC (average precision) — PR-AUC matters especially for rare labels like NAFLD.

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
import xgboost as xgb

MODEL_LABELS = PRIMARY_LABELS  # DXA labels handled separately in Step 11 (smaller n, has NaNs)

for lab in MODEL_LABELS:
    n_nan = df_clean[lab].isna().sum()
    if n_nan > 0:
        print(f"{lab}: {n_nan} rows excluded (NaN label, e.g. NAFLD heavy-drinker exclusion)")

def get_model(name):
    if name == 'LR':
        return LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    if name == 'RF':
        return RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                       random_state=42, n_jobs=-1)
    if name == 'XGB':
        return xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                                  eval_metric='logloss', random_state=42, n_jobs=-1)
    raise ValueError(name)

results = []

for label in MODEL_LABELS:
    valid_train = df_train_p[label].notna()
    valid_test  = df_test_p[label].notna()
    y_train = df_train_p.loc[valid_train, label].astype(int)
    y_test  = df_test_p.loc[valid_test, label].astype(int)
    for tier_name in TIERS:
        feats = tier_features_for_label(label, tier_name)
        X_train = df_train_p.loc[valid_train, feats]
        X_test  = df_test_p.loc[valid_test, feats]
        for model_name in ['LR', 'RF', 'XGB']:
            model = get_model(model_name)
            model.fit(X_train, y_train)
            proba = model.predict_proba(X_test)[:, 1]
            pred  = (proba >= 0.5).astype(int)

            f1  = f1_score(y_test, pred, average='weighted', zero_division=0)
            try:
                auc = roc_auc_score(y_test, proba)
                pr_auc = average_precision_score(y_test, proba)
            except ValueError:
                auc, pr_auc = np.nan, np.nan

            results.append({
                'Label': label.replace('Label_', ''),
                'Tier': tier_name,
                'Model': model_name,
                'N_Features': len(feats),
                'F1': round(f1, 3),
                'ROC_AUC': round(auc, 3) if not np.isnan(auc) else np.nan,
                'PR_AUC': round(pr_auc, 3) if not np.isnan(pr_auc) else np.nan,
            })

results_df = pd.DataFrame(results)
print(f"Total runs: {len(results_df)}  ({len(MODEL_LABELS)} labels x {len(TIERS)} tiers x 3 models)")
results_df.to_csv(os.path.join(PROJECT_PATH, 'tiered_results.csv'), index=False)
results_df.sort_values(['Label', 'Tier', 'Model'])


Label_NAFLD: 1138 rows excluded (NaN label, e.g. NAFLD heavy-drinker exclusion)
Total runs: 105  (7 labels x 5 tiers x 3 models)


,Label,Tier,Model,N_Features,F1,ROC_AUC,PR_AUC
30,Diabetes,T0_TapeMeasureOnly,LR,6,0.726,0.788,0.351
31,Diabetes,T0_TapeMeasureOnly,RF,6,0.793,0.751,0.305
32,Diabetes,T0_TapeMeasureOnly,XGB,6,0.785,0.775,0.344
33,Diabetes,T1_BasicLipids,LR,11,0.742,0.814,0.444
34,Diabetes,T1_BasicLipids,RF,11,0.805,0.802,0.428
...,...,...,...,...,...,...,...
55,NAFLD,T3_ClinicalPanel,RF,18,0.891,0.685,0.234
56,NAFLD,T3_ClinicalPanel,XGB,18,0.891,0.715,0.220
57,NAFLD,T4_EnhancedInstrument,LR,18,0.721,0.685,0.143
58,NAFLD,T4_EnhancedInstrument,RF,18,0.891,0.685,0.234


## Step 11: DXA sub-cohort validation (ages 18-59 only)
Same idea, run only on rows where the DXA label is defined. Reports how the real body-fat-percentage labels compare to the BMI proxy used in the primary cohort.

In [14]:
dxa_results = []

for label in DXA_LABELS:
    valid_train = df_train_p[label].notna()
    valid_test  = df_test_p[label].notna()
    if valid_train.sum() < 50 or valid_test.sum() < 20:
        print(f"Skipping {label}: not enough DXA-valid rows in split "
              f"(train={valid_train.sum()}, test={valid_test.sum()})")
        continue

    y_train = df_train_p.loc[valid_train, label].astype(int)
    y_test  = df_test_p.loc[valid_test, label].astype(int)

    for tier_name in TIERS:
        feats = tier_features_for_label(label, tier_name)
        X_train = df_train_p.loc[valid_train, feats]
        X_test  = df_test_p.loc[valid_test, feats]
        for model_name in ['LR', 'RF', 'XGB']:
            model = get_model(model_name)
            model.fit(X_train, y_train)
            proba = model.predict_proba(X_test)[:, 1]
            pred  = (proba >= 0.5).astype(int)
            f1  = f1_score(y_test, pred, average='weighted', zero_division=0)
            try:
                auc = roc_auc_score(y_test, proba)
                pr_auc = average_precision_score(y_test, proba)
            except ValueError:
                auc, pr_auc = np.nan, np.nan
            dxa_results.append({
                'Label': label.replace('Label_', ''), 'Tier': tier_name, 'Model': model_name,
                'N_train': valid_train.sum(), 'N_test': valid_test.sum(),
                'F1': round(f1, 3), 'ROC_AUC': round(auc, 3) if not np.isnan(auc) else np.nan,
                'PR_AUC': round(pr_auc, 3) if not np.isnan(pr_auc) else np.nan,
            })

dxa_results_df = pd.DataFrame(dxa_results)
dxa_results_df.to_csv(os.path.join(PROJECT_PATH, 'dxa_subcohort_results.csv'), index=False)
dxa_results_df


,Label,Tier,Model,N_train,N_test,F1,ROC_AUC,PR_AUC
0,HighBodyFat_DXA,T0_TapeMeasureOnly,LR,1956,496,0.860,0.941,0.969
1,HighBodyFat_DXA,T0_TapeMeasureOnly,RF,1956,496,0.866,0.929,0.964
2,HighBodyFat_DXA,T0_TapeMeasureOnly,XGB,1956,496,0.859,0.936,0.962
3,HighBodyFat_DXA,T1_BasicLipids,LR,1956,496,0.866,0.941,0.971
4,HighBodyFat_DXA,T1_BasicLipids,RF,1956,496,0.866,0.939,0.973
5,HighBodyFat_DXA,T1_BasicLipids,XGB,1956,496,0.864,0.933,0.969
6,HighBodyFat_DXA,T2_ExtendedMetabolic,LR,1956,496,0.866,0.940,0.968
7,HighBodyFat_DXA,T2_ExtendedMetabolic,RF,1956,496,0.864,0.940,0.973
8,HighBodyFat_DXA,T2_ExtendedMetabolic,XGB,1956,496,0.864,0.935,0.970
9,HighBodyFat_DXA,T3_ClinicalPanel,LR,1956,496,0.862,0.940,0.967


## Step 12: Summary tables
Two views: best model per (label, tier), and the tier-by-tier degradation curve for XGBoost specifically (usually the strongest baseline).

In [15]:
pivot_auc = results_df.pivot_table(index='Label', columns=['Tier', 'Model'], values='ROC_AUC')
tier_order = list(TIERS.keys())
pivot_auc = pivot_auc.reindex(columns=pd.MultiIndex.from_product([tier_order, ['LR','RF','XGB']]))
print("=== ROC-AUC by Label x Tier x Model ===")
pivot_auc


=== ROC-AUC by Label x Tier x Model ===


T0_TapeMeasureOnly               T1_BasicLipids         \
                                  LR     RF    XGB             LR     RF   
Label                                                                      
Diabetes                       0.788  0.751  0.775          0.814  0.802   
HighBodyFat_BMI                0.964  0.963  0.966          0.965  0.964   
Hypertension                   0.714  0.686  0.698          0.716  0.701   
InsulinResistance              0.673  0.642  0.660          0.723  0.938   
Low_HDL                        0.692  0.629  0.665          0.808  0.778   
MetSyn                         0.743  0.704  0.725          0.745  0.813   
NAFLD                          0.668  0.647  0.723          0.662  0.698   

                         T2_ExtendedMetabolic               T3_ClinicalPanel  \
                     XGB                   LR     RF    XGB               LR   
Label                                                                          
Diabetes           0.811                0.822  0.812  0.832            0.824   
HighBodyFat_BMI    0.968                0.965  0.963  0.970            0.965   
Hypertension       0.704                0.718  0.710  0.721            0.719   
InsulinResistance  0.939                0.728  0.939  0.943            0.737   
Low_HDL            0.822                0.818  0.767  0.825            0.820   
MetSyn             0.833                0.798  0.843  0.850            0.799   
NAFLD              0.739                0.697  0.703  0.733            0.685   

                                T4_EnhancedInstrument                
                      RF    XGB                    LR     RF    XGB  
Label                                                                
Diabetes           0.831  0.841                 0.838  0.839  0.847  
HighBodyFat_BMI    0.965  0.969                 0.965  0.966  0.969  
Hypertension       0.706  0.718                 0.716  0.709  0.719  
InsulinResistance  0.943  0.945                 0.738  0.943  0.946  
Low_HDL            0.791  0.837                 0.816  0.794  0.829  
MetSyn             0.852  0.855                 0.803  0.857  0.857  
NAFLD              0.685  0.715                 0.685  0.685  0.715

In [16]:
print("=== XGBoost: AUC by tier (the 'how much does more info buy you' curve) ===")
xgb_only = results_df[results_df['Model'] == 'XGB'].pivot_table(
    index='Label', columns='Tier', values='ROC_AUC'
).reindex(columns=tier_order)
xgb_only


=== XGBoost: AUC by tier (the 'how much does more info buy you' curve) ===


Tier,T0_TapeMeasureOnly,T1_BasicLipids,T2_ExtendedMetabolic,T3_ClinicalPanel,T4_EnhancedInstrument
Label,,,,,
Diabetes,0.775,0.811,0.832,0.841,0.847
HighBodyFat_BMI,0.966,0.968,0.970,0.969,0.969
Hypertension,0.698,0.704,0.721,0.718,0.719
InsulinResistance,0.660,0.939,0.943,0.945,0.946
Low_HDL,0.665,0.822,0.825,0.837,0.829
MetSyn,0.725,0.833,0.850,0.855,0.857
NAFLD,0.723,0.739,0.733,0.715,0.715


## Step 12b: Positive-class diagnostics (sensitivity, specificity, PPV, NPV)
Weighted F1 is dominated by the majority (negative) class for rare labels -- it can look respectable while positive-class recall is weak. This reports the metrics that actually matter for a screening tool, at the T3_ClinicalPanel tier (XGBoost), for every label, with special attention to the rarer ones (NAFLD, Diabetes).

In [17]:
from sklearn.metrics import confusion_matrix

diag_rows = []
for label in PRIMARY_LABELS:
    valid_train = df_train_p[label].notna()
    valid_test  = df_test_p[label].notna()
    y_train = df_train_p.loc[valid_train, label].astype(int)
    y_test  = df_test_p.loc[valid_test, label].astype(int)

    feats = tier_features_for_label(label, 'T3_ClinicalPanel')
    X_train = df_train_p.loc[valid_train, feats]
    X_test  = df_test_p.loc[valid_test, feats]

    model = get_model('XGB')
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan   # recall / TPR
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan   # TNR
    ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan           # precision
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
    pr_auc = average_precision_score(y_test, proba)
    prevalence = y_test.mean()

    diag_rows.append({
        'Label': label.replace('Label_', ''),
        'Prevalence': round(prevalence, 3),
        'Sensitivity': round(sensitivity, 3),
        'Specificity': round(specificity, 3),
        'PPV': round(ppv, 3),
        'NPV': round(npv, 3),
        'PR_AUC': round(pr_auc, 3),
        'TP': tp, 'FN': fn, 'FP': fp, 'TN': tn,
    })

diag_df = pd.DataFrame(diag_rows)
diag_df.to_csv(os.path.join(PROJECT_PATH, 'positive_class_diagnostics.csv'), index=False)
print("=== Positive-class diagnostics, T3_ClinicalPanel / XGBoost, threshold=0.5 ===")
print("(Flag anything with low Sensitivity or low PR_AUC despite a decent ROC-AUC above --")
print(" that's exactly what weighted F1 can hide.)")
diag_df


=== Positive-class diagnostics, T3_ClinicalPanel / XGBoost, threshold=0.5 ===
(Flag anything with low Sensitivity or low PR_AUC despite a decent ROC-AUC above --
 that's exactly what weighted F1 can hide.)


,Label,Prevalence,Sensitivity,Specificity,PPV,NPV,PR_AUC,TP,FN,FP,TN
0,HighBodyFat_BMI,0.404,0.859,0.930,0.893,0.907,0.961,384,63,46,614
1,Low_HDL,0.276,0.408,0.930,0.691,0.805,0.692,125,181,56,745
2,Diabetes,0.155,0.297,0.973,0.671,0.883,0.520,51,121,25,910
3,NAFLD,0.074,0.000,0.999,0.000,0.926,0.220,0,65,1,818
4,Hypertension,0.407,0.633,0.693,0.585,0.734,0.566,285,165,202,455
5,InsulinResistance,0.235,0.754,0.927,0.760,0.925,0.837,196,64,62,785
6,MetSyn,0.222,0.459,0.931,0.657,0.858,0.659,113,133,59,802


## Step 12c: Insulin Resistance ablation ladder
IR's tier-experiment AUC jumps from ~0.68 (T0) to ~0.95 (T1+) once TG/HDL/VAI/LAP enter the feature set. `LEAKAGE_EXCLUSIONS['Label_InsulinResistance']` already removes HOMA_IR, Glucose, Insulin, and TyG at every tier -- so this isn't glucose/insulin leakage sneaking back in. This ladder isolates *which* remaining feature actually drives the jump, adding one feature at a time on top of anthropometry.

In [18]:
label = 'Label_InsulinResistance'
valid_train = df_train_p[label].notna()
valid_test  = df_test_p[label].notna()
y_train = df_train_p.loc[valid_train, label].astype(int)
y_test  = df_test_p.loc[valid_test, label].astype(int)

base = ['Gender', 'Age', 'BMI', 'Waist', 'Height', 'WHtR']
ladder_steps = [
    ('Anthropometry only',        base),
    ('+ Triglycerides',           base + ['Triglycerides']),
    ('+ HDL',                     base + ['Triglycerides', 'HDL']),
    ('+ VAI',                     base + ['Triglycerides', 'HDL', 'VAI']),
    ('+ LAP',                     base + ['Triglycerides', 'HDL', 'VAI', 'LAP']),
    ('+ LDL (T1 full)',           base + ['Triglycerides', 'HDL', 'VAI', 'LAP', 'LDL']),
]

ladder_rows = []
for step_name, feats in ladder_steps:
    # respect the label's own leakage exclusions even though none of these are excluded for IR
    feats = leakage_free_features(label, feats)
    X_train = df_train_p.loc[valid_train, feats]
    X_test  = df_test_p.loc[valid_test, feats]
    model = get_model('XGB')
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)
    ladder_rows.append({'Step': step_name, 'N_Features': len(feats), 'ROC_AUC': round(auc, 3)})

ladder_df = pd.DataFrame(ladder_rows)
ladder_df.to_csv(os.path.join(PROJECT_PATH, 'ir_ablation_ladder.csv'), index=False)
print("=== Insulin Resistance: which feature drives the T0 -> T1 AUC jump? ===")
ladder_df


=== Insulin Resistance: which feature drives the T0 -> T1 AUC jump? ===


,Step,N_Features,ROC_AUC
0,Anthropometry only,6,0.660
1,+ Triglycerides,7,0.937
2,+ HDL,8,0.937
3,+ VAI,9,0.940
4,+ LAP,10,0.940
5,+ LDL (T1 full),11,0.939


## Step 12d: 5-fold cross-validation (T3_ClinicalPanel, XGBoost only)
Running full 5-fold CV across all 9 labels x 5 tiers x 3 models (540 fits) is overkill for a course deliverable. This scopes CV to the primary reported result -- T3_ClinicalPanel, XGBoost -- to get a confidence interval on the headline numbers. LR/RF stay single-split since they're baselines, not the finding being defended.

In [19]:
from sklearn.model_selection import StratifiedKFold

cv_rows = []
for label in PRIMARY_LABELS:
    valid = df_clean[label].notna()
    y_all = df_clean.loc[valid, label].astype(int)
    feats = tier_features_for_label(label, 'T3_ClinicalPanel')

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_aucs, fold_f1s = [], []
    idx_valid = df_clean.loc[valid].index.values

    # Fit preprocessing (median/winsorize) fresh within each fold's training data
    for fold_train_pos, fold_test_pos in skf.split(idx_valid, y_all):
        fold_train_idx = idx_valid[fold_train_pos]
        fold_test_idx  = idx_valid[fold_test_pos]

        fold_train_raw = df_clean.loc[fold_train_idx]
        fold_test_raw  = df_clean.loc[fold_test_idx]

        f_medians, f_winsor = fit_preprocessing(fold_train_raw, ALL_FEATURES)
        fold_train_p = apply_preprocessing(fold_train_raw, f_medians, f_winsor)
        fold_test_p  = apply_preprocessing(fold_test_raw, f_medians, f_winsor)

        X_tr = fold_train_p[feats]
        X_te = fold_test_p[feats]
        y_tr = fold_train_p[label].astype(int)
        y_te = fold_test_p[label].astype(int)

        model = get_model('XGB')
        model.fit(X_tr, y_tr)
        proba = model.predict_proba(X_te)[:, 1]
        pred = (proba >= 0.5).astype(int)

        try:
            fold_aucs.append(roc_auc_score(y_te, proba))
        except ValueError:
            pass
        fold_f1s.append(f1_score(y_te, pred, average='weighted', zero_division=0))

    cv_rows.append({
        'Label': label.replace('Label_', ''),
        'CV_AUC_Mean': round(np.mean(fold_aucs), 3),
        'CV_AUC_Std': round(np.std(fold_aucs), 3),
        'CV_F1_Mean': round(np.mean(fold_f1s), 3),
        'CV_F1_Std': round(np.std(fold_f1s), 3),
    })

cv_df = pd.DataFrame(cv_rows)
cv_df.to_csv(os.path.join(PROJECT_PATH, 'cv_t3_xgb.csv'), index=False)
print("=== 5-fold CV, T3_ClinicalPanel, XGBoost (mean +/- std) ===")
cv_df


=== 5-fold CV, T3_ClinicalPanel, XGBoost (mean +/- std) ===


,Label,CV_AUC_Mean,CV_AUC_Std,CV_F1_Mean,CV_F1_Std
0,HighBodyFat_BMI,0.965,0.004,0.894,0.008
1,Low_HDL,0.843,0.004,0.775,0.010
2,Diabetes,0.821,0.011,0.831,0.008
3,NAFLD,0.699,0.033,0.894,0.003
4,Hypertension,0.701,0.013,0.645,0.008
5,InsulinResistance,0.956,0.003,0.892,0.006
6,MetSyn,0.861,0.011,0.829,0.007


## Step 12e: Threshold tuning (on a validation split carved from TRAIN, not test)
The 0.5 cutoff used everywhere above is a default, not a chosen operating point. This carves a validation split out of the training data only, sweeps thresholds there to maximize F1, then reports what that tuned threshold does on the untouched test set -- test set AUC/ranking is unaffected since threshold choice doesn't change probability ranking, only the F1/sensitivity tradeoff at the cutoff actually used for a positive call.

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score as _f1

threshold_rows = []
for label in PRIMARY_LABELS:
    valid_train = df_train_p[label].notna()
    y_train_full = df_train_p.loc[valid_train, label].astype(int)
    feats = tier_features_for_label(label, 'T3_ClinicalPanel')
    X_train_full = df_train_p.loc[valid_train, feats]

    # carve val out of train only -- test set is never touched by threshold selection
    X_tr2, X_val, y_tr2, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.2, random_state=42,
        stratify=y_train_full if y_train_full.nunique() > 1 else None
    )

    model = get_model('XGB')
    model.fit(X_tr2, y_tr2)
    val_proba = model.predict_proba(X_val)[:, 1]

    thresholds = np.arange(0.05, 0.96, 0.01)
    best_thresh, best_f1 = 0.5, -1
    for t in thresholds:
        f1_t = _f1(y_val, (val_proba >= t).astype(int), average='weighted', zero_division=0)
        if f1_t > best_f1:
            best_f1, best_thresh = f1_t, t

    # refit on FULL train, evaluate tuned threshold on held-out test
    model_full = get_model('XGB')
    model_full.fit(X_train_full, y_train_full)
    valid_test = df_test_p[label].notna()
    y_test = df_test_p.loc[valid_test, label].astype(int)
    X_test_l = df_test_p.loc[valid_test, feats]
    test_proba = model_full.predict_proba(X_test_l)[:, 1]

    f1_default = _f1(y_test, (test_proba >= 0.5).astype(int), average='weighted', zero_division=0)
    f1_tuned = _f1(y_test, (test_proba >= best_thresh).astype(int), average='weighted', zero_division=0)

    threshold_rows.append({
        'Label': label.replace('Label_', ''),
        'Tuned_Threshold': round(best_thresh, 2),
        'Test_F1_at_0.5': round(f1_default, 3),
        'Test_F1_at_Tuned': round(f1_tuned, 3),
        'F1_Gain': round(f1_tuned - f1_default, 3),
    })

threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_csv(os.path.join(PROJECT_PATH, 'tuned_thresholds.csv'), index=False)
print("=== Threshold tuned on train-internal validation split, evaluated on untouched test ===")
threshold_df

=== Threshold tuned on train-internal validation split, evaluated on untouched test ===


,Label,Tuned_Threshold,Test_F1_at_0.5,Test_F1_at_Tuned,F1_Gain
0,HighBodyFat_BMI,0.46,0.901,0.905,0.004
1,Low_HDL,0.40,0.766,0.777,0.011
2,Diabetes,0.38,0.846,0.848,0.002
3,NAFLD,0.30,0.891,0.903,0.012
4,Hypertension,0.43,0.670,0.662,-0.008
5,InsulinResistance,0.32,0.886,0.885,-0.001
6,MetSyn,0.48,0.815,0.817,0.002


## Step 13: Multi-Task Neural Network — leakage-safe via per-label input masking
**The problem:** a standard MTL-NN feeds ONE shared input vector to all heads. But your leakage exclusions differ per label (Diabetes excludes Glucose; MetSyn excludes Waist/TG/HDL/BP; etc.) — a naive shared-input model would leak for most labels.

**The fix:** each label gets its own binary mask (1 = safe, 0 = excluded for that label) applied to the T3 full-feature input BEFORE a small label-specific adapter layer, which then feeds into a genuinely SHARED trunk (same weights, reused across all labels). This preserves the cross-disease representation-learning that's the whole point of MTL, without any head seeing its own label-defining features.

NAFLD's NaN rows (heavy-drinker exclusion) are handled via per-output `sample_weight=0` — those rows still flow through the network (all heads share one batch) but contribute zero gradient to the NAFLD head specifically.

In [22]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.preprocessing import StandardScaler

T3_FEATURES = TIERS['T3_ClinicalPanel']

# Per-label binary masks over T3_FEATURES
label_masks = {}
for label in PRIMARY_LABELS:
    safe = set(leakage_free_features(label, T3_FEATURES))
    mask = np.array([1.0 if f in safe else 0.0 for f in T3_FEATURES], dtype='float32')
    label_masks[label] = mask
    print(f"{label:<30} keeps {int(mask.sum())}/{len(T3_FEATURES)} features")

# Scale on train only
scaler_mtl = StandardScaler()
X_train_mtl = scaler_mtl.fit_transform(df_train_p[T3_FEATURES]).astype('float32')
X_test_mtl  = scaler_mtl.transform(df_test_p[T3_FEATURES]).astype('float32')

n_features = len(T3_FEATURES)

# y arrays + sample weights (0 for NaN-label rows, filled with placeholder 0 so shapes align)
y_train_dict, y_test_dict = {}, {}
sw_train_dict, sw_test_dict = {}, {}
for label in PRIMARY_LABELS:
    name = label.replace('Label_', '')
    yt = df_train_p[label]
    yv = df_test_p[label]
    sw_train_dict[name] = yt.notna().astype('float32').values
    sw_test_dict[name]  = yv.notna().astype('float32').values
    y_train_dict[name] = yt.fillna(0).astype('float32').values
    y_test_dict[name]  = yv.fillna(0).astype('float32').values


Label_HighBodyFat_BMI          keeps 19/20 features
Label_Low_HDL                  keeps 19/20 features
Label_Diabetes                 keeps 16/20 features
Label_NAFLD                    keeps 18/20 features
Label_Hypertension             keeps 18/20 features
Label_InsulinResistance        keeps 16/20 features
Label_MetSyn                   keeps 9/20 features


In [23]:
def build_masked_mtl(n_features, labels, label_masks):
    inputs = keras.Input(shape=(n_features,), name='input')

    # Shared trunk — SAME layer objects reused for every label (true weight sharing)
    shared_l1 = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-3))
    shared_bn1 = layers.BatchNormalization()
    shared_do1 = layers.Dropout(0.3)
    shared_l2 = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-3))
    shared_bn2 = layers.BatchNormalization()

    outputs = []
    for label in labels:
        name = label.replace('Label_', '')
        mask_const = tf.constant(label_masks[label])
        masked = layers.Lambda(lambda x, m=mask_const: x * m,
                                 name=f'mask_{name}')(inputs)
        adapter = layers.Dense(32, activation='relu',
                                 name=f'adapter_{name}')(masked)
        x = shared_l1(adapter)
        x = shared_bn1(x)
        x = shared_do1(x)
        x = shared_l2(x)
        x = shared_bn2(x)
        head = layers.Dense(16, activation='relu', name=f'head_{name}')(x)
        out = layers.Dense(1, activation='sigmoid', name=name)(head)
        outputs.append(out)

    return keras.Model(inputs=inputs, outputs=outputs)

mtl_model = build_masked_mtl(n_features, PRIMARY_LABELS, label_masks)

loss_dict = {label.replace('Label_', ''): 'binary_crossentropy' for label in PRIMARY_LABELS}
metrics_dict = {label.replace('Label_', ''): 'AUC' for label in PRIMARY_LABELS}

mtl_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                   loss=loss_dict, metrics=metrics_dict)

callbacks = [
    keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor='val_loss'),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, verbose=0),
]

# Convert dictionaries to lists, ensuring the order matches PRIMARY_LABELS
y_train_list = [y_train_dict[label.replace('Label_', '')] for label in PRIMARY_LABELS]
sw_train_list = [sw_train_dict[label.replace('Label_', '')] for label in PRIMARY_LABELS]
y_test_list = [y_test_dict[label.replace('Label_', '')] for label in PRIMARY_LABELS]
sw_test_list = [sw_test_dict[label.replace('Label_', '')] for label in PRIMARY_LABELS]

history = mtl_model.fit(
    X_train_mtl, y_train_list,
    sample_weight=sw_train_list,
    validation_data=(X_test_mtl, y_test_list, sw_test_list),
    epochs=100, batch_size=64, callbacks=callbacks, verbose=1,
)

Epoch 1/100
70/70 ━━━━━━━━━━━━━━━━━━━━ 24s 57ms/step - Diabetes_AUC: 0.6156 - Diabetes_loss: 0.7495 - HighBodyFat_BMI_AUC: 0.8097 - HighBodyFat_BMI_loss: 0.5393 - Hypertension_AUC: 0.6061 - Hypertension_loss: 0.6711 - InsulinResistance_AUC: 0.6503 - InsulinResistance_loss: 0.7958 - Low_HDL_AUC: 0.6146 - Low_HDL_loss: 0.6982 - MetSyn_AUC: 0.5916 - MetSyn_loss: 0.7951 - NAFLD_AUC: 0.5175 - NAFLD_loss: 0.5170 - loss: 4.8657 - val_Diabetes_AUC: 0.7639 - val_Diabetes_loss: 0.4658 - val_HighBodyFat_BMI_AUC: 0.9205 - val_HighBodyFat_BMI_loss: 0.3791 - val_Hypertension_AUC: 0.6950 - val_Hypertension_loss: 0.6100 - val_InsulinResistance_AUC: 0.7615 - val_InsulinResistance_loss: 0.5727 - val_Low_HDL_AUC: 0.7152 - val_Low_HDL_loss: 0.5383 - val_MetSyn_AUC: 0.7127 - val_MetSyn_loss: 0.5601 - val_NAFLD_AUC: 0.5966 - val_NAFLD_loss: 0.3187 - val_loss: 3.5687 - learning_rate: 0.0010
Epoch 2/100
70/70 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - Diabetes_AUC: 0.7170 - Diabetes_loss: 0.4508 - HighBodyFat_BMI_AU

In [24]:
preds = mtl_model.predict(X_test_mtl, verbose=0)

mtl_rows = []
for i, label in enumerate(PRIMARY_LABELS):
    name = label.replace('Label_', '')
    valid = sw_test_dict[name].astype(bool)
    y_true = y_test_dict[name][valid]
    y_prob = preds[i].flatten()[valid]
    pred_binary = (y_prob >= 0.5).astype(int)
    f1 = f1_score(y_true, pred_binary, average='weighted', zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_prob)
        pr_auc = average_precision_score(y_true, y_prob)
    except ValueError:
        auc, pr_auc = np.nan, np.nan
    mtl_rows.append({'Label': name, 'MTL_F1': round(f1, 3),
                      'MTL_AUC': round(auc, 3) if not np.isnan(auc) else np.nan,
                      'MTL_PR_AUC': round(pr_auc, 3) if not np.isnan(pr_auc) else np.nan})

mtl_results_df = pd.DataFrame(mtl_rows)
mtl_results_df.to_csv(os.path.join(PROJECT_PATH, 'mtl_results.csv'), index=False)
print("=== MTL-NN (leakage-safe, masked shared backbone) — T3 full panel ===")
mtl_results_df


=== MTL-NN (leakage-safe, masked shared backbone) — T3 full panel ===


,Label,MTL_F1,MTL_AUC,MTL_PR_AUC
0,HighBodyFat_BMI,0.884,0.963,0.953
1,Low_HDL,0.770,0.821,0.694
2,Diabetes,0.831,0.831,0.517
3,NAFLD,0.893,0.707,0.170
4,Hypertension,0.629,0.698,0.564
5,InsulinResistance,0.874,0.928,0.800
6,MetSyn,0.801,0.843,0.627


### Step 13b: Does MTL actually help? (single-task NN comparison)
Same masked-input idea, but each label gets its OWN dedicated network — no shared trunk. If MTL-NN beats this on a given label, sharing genuinely helped that label. If not, that label doesn't benefit from multi-task sharing (which is a legitimate and interesting finding, not a failure).

In [25]:
single_task_rows = []

for label in PRIMARY_LABELS:
    name = label.replace('Label_', '')
    mask = label_masks[label]
    valid_train = sw_train_dict[name].astype(bool)
    valid_test = sw_test_dict[name].astype(bool)

    X_tr = X_train_mtl[valid_train] * mask
    X_te = X_test_mtl[valid_test] * mask
    y_tr = y_train_dict[name][valid_train]
    y_te = y_test_dict[name][valid_test]

    inputs = keras.Input(shape=(n_features,))
    x = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-3))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-3))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(16, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    st_model = keras.Model(inputs, out)
    st_model.compile(optimizer=keras.optimizers.Adam(0.001), loss='binary_crossentropy', metrics=['AUC'])
    st_model.fit(X_tr, y_tr, validation_data=(X_te, y_te), epochs=100, batch_size=64,
                 callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True,
                                                           monitor='val_loss')],
                 verbose=0)

    y_prob = st_model.predict(X_te, verbose=0).flatten()
    pred_binary = (y_prob >= 0.5).astype(int)
    f1 = f1_score(y_te, pred_binary, average='weighted', zero_division=0)
    try:
        auc = roc_auc_score(y_te, y_prob)
    except ValueError:
        auc = np.nan
    single_task_rows.append({'Label': name, 'SingleTask_F1': round(f1, 3),
                              'SingleTask_AUC': round(auc, 3) if not np.isnan(auc) else np.nan})
    print(f"{name:<25} single-task AUC: {auc:.3f}" if not np.isnan(auc) else f"{name}: AUC undefined")

single_task_df = pd.DataFrame(single_task_rows)
mtl_vs_single = mtl_results_df.merge(single_task_df, on='Label')
mtl_vs_single['MTL_Advantage'] = mtl_vs_single['MTL_AUC'] - mtl_vs_single['SingleTask_AUC']
mtl_vs_single.to_csv(os.path.join(PROJECT_PATH, 'mtl_vs_singletask.csv'), index=False)
print("\n=== Does sharing help? (positive = MTL beats single-task) ===")
mtl_vs_single[['Label', 'SingleTask_AUC', 'MTL_AUC', 'MTL_Advantage']]


HighBodyFat_BMI           single-task AUC: 0.963
Low_HDL                   single-task AUC: 0.868
Diabetes                  single-task AUC: 0.837
NAFLD                     single-task AUC: 0.691
Hypertension              single-task AUC: 0.711
InsulinResistance         single-task AUC: 0.941
MetSyn                    single-task AUC: 0.857

=== Does sharing help? (positive = MTL beats single-task) ===


,Label,SingleTask_AUC,MTL_AUC,MTL_Advantage
0,HighBodyFat_BMI,0.963,0.963,0.000
1,Low_HDL,0.868,0.821,-0.047
2,Diabetes,0.837,0.831,-0.006
3,NAFLD,0.691,0.707,0.016
4,Hypertension,0.711,0.698,-0.013
5,InsulinResistance,0.941,0.928,-0.013
6,MetSyn,0.857,0.843,-0.014


## Step 14: Missing-test robustness (degradation curve)
Trains XGBoost per label on **raw, un-imputed** T3 features (XGBoost routes NaN natively — no imputation needed), then at test time randomly masks an increasing fraction of features to simulate a patient missing that many tests. This uses real native missing-value handling, not a median-imputed stand-in, so it reflects genuine model behavior on incomplete data.

In [26]:
df_train_raw = df_train.copy()  # NOT imputed — XGBoost handles NaN directly
df_test_raw  = df_test.copy()

def train_xgb_raw(label, tier_name='T3_ClinicalPanel'):
    feats = tier_features_for_label(label, tier_name)
    valid = df_train_raw[label].notna()
    y = df_train_raw.loc[valid, label].astype(int)
    X = df_train_raw.loc[valid, feats]
    model = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                               eval_metric='logloss', random_state=42, n_jobs=-1)
    model.fit(X, y)
    return model, feats

def eval_with_missingness(model, feats, label, missing_frac, seed=42):
    valid = df_test_raw[label].notna()
    y = df_test_raw.loc[valid, label].astype(int)
    X = df_test_raw.loc[valid, feats].copy()
    if missing_frac > 0:
        rng = np.random.default_rng(seed)
        drop_mask = rng.random(X.shape) < missing_frac
        X = X.mask(drop_mask)
    proba = model.predict_proba(X)[:, 1]
    try:
        return roc_auc_score(y, proba)
    except ValueError:
        return np.nan

MISSING_FRACS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
degradation_rows = []

for label in PRIMARY_LABELS:
    model, feats = train_xgb_raw(label)
    for frac in MISSING_FRACS:
        auc = eval_with_missingness(model, feats, label, frac)
        degradation_rows.append({
            'Label': label.replace('Label_', ''),
            'Missing_Frac': frac,
            'AUC': round(auc, 3) if not np.isnan(auc) else np.nan,
        })

degradation_df = pd.DataFrame(degradation_rows)
degradation_pivot = degradation_df.pivot(index='Label', columns='Missing_Frac', values='AUC')
degradation_pivot.to_csv(os.path.join(PROJECT_PATH, 'missingness_degradation.csv'))
print("=== AUC vs. fraction of features randomly missing at test time (XGBoost, native NaN handling) ===")
degradation_pivot


=== AUC vs. fraction of features randomly missing at test time (XGBoost, native NaN handling) ===


Missing_Frac,0.0,0.1,0.2,0.3,0.4,0.5
Label,,,,,,
Diabetes,0.842,0.826,0.799,0.772,0.732,0.715
HighBodyFat_BMI,0.972,0.947,0.923,0.884,0.849,0.805
Hypertension,0.708,0.669,0.630,0.607,0.600,0.590
InsulinResistance,0.947,0.927,0.907,0.890,0.873,0.841
Low_HDL,0.842,0.794,0.754,0.698,0.666,0.633
MetSyn,0.855,0.817,0.784,0.772,0.757,0.731
NAFLD,0.714,0.645,0.628,0.600,0.612,0.548


## Step 15: Final combined summary
Everything in one place — worth pasting back for review.

In [27]:
print("### 1. Best model per label (from Step 10 tiered results, T3 tier) ###")
t3 = results_df[results_df['Tier'] == 'T3_ClinicalPanel']
print(t3.loc[t3.groupby('Label')['ROC_AUC'].idxmax()][['Label', 'Model', 'ROC_AUC', 'PR_AUC', 'F1']])

print("\n### 2. Tier degradation (XGBoost) ###")
print(xgb_only)

print("\n### 3. Positive-class diagnostics (T3, XGBoost) ###")
print(diag_df)

print("\n### 4. Insulin Resistance ablation ladder ###")
print(ladder_df)

print("\n### 5. 5-fold CV (T3, XGBoost) ###")
print(cv_df)

print("\n### 6. Tuned decision thresholds ###")
print(threshold_df)

print("\n### 7. MTL vs single-task ###")
print(mtl_vs_single[['Label', 'SingleTask_AUC', 'MTL_AUC', 'MTL_Advantage']])

print("\n### 8. Missingness robustness ###")
print(degradation_pivot)

print("\n### 9. DXA sub-cohort (real body-fat labels, ages 18-59) ###")
print(dxa_results_df[dxa_results_df['Model'] == 'XGB'][['Label', 'Tier', 'N_test', 'ROC_AUC']])


### 1. Best model per label (from Step 10 tiered results, T3 tier) ###
                 Label Model  ROC_AUC  PR_AUC     F1
41            Diabetes   XGB    0.841   0.520  0.846
11     HighBodyFat_BMI   XGB    0.969   0.961  0.901
69        Hypertension    LR    0.719   0.573  0.654
86   InsulinResistance   XGB    0.945   0.837  0.886
26             Low_HDL   XGB    0.837   0.692  0.766
101             MetSyn   XGB    0.855   0.659  0.815
56               NAFLD   XGB    0.715   0.220  0.891

### 2. Tier degradation (XGBoost) ###
Tier               T0_TapeMeasureOnly  T1_BasicLipids  T2_ExtendedMetabolic  \
Label                                                                         
Diabetes                        0.775           0.811                 0.832   
HighBodyFat_BMI                 0.966           0.968                 0.970   
Hypertension                    0.698           0.704                 0.721   
InsulinResistance               0.660           0.939                 0